In [ ]:
import pandas as pd

df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")  # remplace par le vrai nom de fichier

# Lignes où au moins un sous-score touche 0 (ou proche de 0)
sous_scores = ["score_temperature", "score_pluie", "score_ph"]
mask_zero = (df[sous_scores] <= 0.01).any(axis=1)

print(f"{mask_zero.sum()} lignes avec au moins un sous-score à 0")
print(df.loc[mask_zero, ["maille_id", "culture", "annee"] + sous_scores + ["score_global"]].head(20))

# Vérifier aussi le cas limite : sous-score négatif (si l'interpolation linéaire
# n'est pas clampée, une valeur très hors bornes absolues donnerait un score < 0)
mask_negatif = (df[sous_scores] < 0).any(axis=1)
print(f"\n{mask_negatif.sum()} lignes avec un sous-score négatif (si > 0, la formule n'est PAS clampée)")
if mask_negatif.sum() > 0:
    print(df.loc[mask_negatif, ["maille_id", "culture", "annee"] + sous_scores])

In [ ]:
import pandas as pd

In [ ]:
# Charger le CSV
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")

# Pour la vigne, supposons que les bornes absolues sont 
# (à faire confirmer par Ugo) : pluie_absolue_min_mm_an = 300, pluie_absolue_max_mm_an = 1500
# Filtre vigne, score_pluie == 0
vigne_zero = df[(df["culture"] == "Vigne") & (df["score_pluie"] == 0)]

# Afficher les précipitations min et max dans ces cas
print("Précipitations des vignes avec score_pluie = 0 :")
print(vigne_zero["precipitations_annuelles_mm"].describe())

# Vérifier qu'elles sont bien toutes en dehors de [300, 1500] (ou autres bornes)
# Si certaines sont dans l'intervalle, alors il y a un problème.

In [ ]:
print(df.columns.tolist())

In [ ]:
vigne = df[df["culture"] == "Vigne"]
print("Bornes absolues min (uniques) :", vigne["pluie_absolue_min_mm_an"].unique())
print("Bornes absolues max (uniques) :", vigne["pluie_absolue_max_mm_an"].unique())
print("Bornes optimales min (uniques) :", vigne["pluie_optimale_min_mm_an"].unique())
print("Bornes optimales max (uniques) :", vigne["pluie_optimale_max_mm_an"].unique())

In [ ]:
# Échantillon de 10 lignes vigne avec score_pluie = 0
sample = df[(df["culture"] == "Vigne") & (df["score_pluie"] == 0)].sample(10, random_state=42)

def compute_linear_score(x, abs_min, opt_min, opt_max, abs_max):
    if x <= abs_min or x >= abs_max:
        return 0.0
    if opt_min <= x <= opt_max:
        return 100.0
    if abs_min < x < opt_min:
        return (x - abs_min) / (opt_min - abs_min) * 100
    if opt_max < x < abs_max:
        return (abs_max - x) / (abs_max - opt_max) * 100
    return 0.0

for _, row in sample.iterrows():
    x = row["precipitations_annuelles_mm"]
    score_calc = compute_linear_score(
        x,
        row["pluie_absolue_min_mm_an"],
        row["pluie_optimale_min_mm_an"],
        row["pluie_optimale_max_mm_an"],
        row["pluie_absolue_max_mm_an"]
    )
    print(f"maille {row['maille_id']} annee {row['annee']} -> precip={x:.1f}, score reel={row['score_pluie']}, score recalcule={score_calc:.2f}")

In [ ]:
vigne_zero = df[(df["culture"] == "Vigne") & (df["score_pluie"] == 0)]
vigne_pos = df[(df["culture"] == "Vigne") & (df["score_pluie"] > 0)]
print("Stats precip pour score_pluie = 0 :")
print(vigne_zero["precipitations_annuelles_mm"].describe())
print("\nStats precip pour score_pluie > 0 :")
print(vigne_pos["precipitations_annuelles_mm"].describe())

In [ ]:
#copie corrigée

import pandas as pd

# Charger le CSV original
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")

# Définir les nouvelles bornes pour la vigne
new_abs_min = 400      # inchangé
new_abs_max = 1500     # modifié (était 1200)
new_opt_min = 600      # légèrement abaissé
new_opt_max = 900      # élargi

# Fonction de score (identique à celle d'Ugo)
def compute_linear_score(x, abs_min, opt_min, opt_max, abs_max):
    if x <= abs_min or x >= abs_max:
        return 0.0
    if opt_min <= x <= opt_max:
        return 100.0
    if abs_min < x < opt_min:
        return (x - abs_min) / (opt_min - abs_min) * 100
    if opt_max < x < abs_max:
        return (abs_max - x) / (abs_max - opt_max) * 100
    return 0.0

# Appliquer la correction uniquement sur les lignes Vigne
mask_vigne = df["culture"] == "Vigne"
df.loc[mask_vigne, "score_pluie_corr"] = df.loc[mask_vigne, "precipitations_annuelles_mm"].apply(
    lambda x: compute_linear_score(x, new_abs_min, new_opt_min, new_opt_max, new_abs_max)
)

# Recalculer le score global avec les mêmes poids (0.6 T, 0.2 P, 0.2 pH)
df["score_global_corr"] = df["score_global"]  # recopie pour les autres cultures
df.loc[mask_vigne, "score_global_corr"] = (
    0.6 * df.loc[mask_vigne, "score_temperature"] +
    0.2 * df.loc[mask_vigne, "score_pluie_corr"] +
    0.2 * df.loc[mask_vigne, "score_ph"]
)

# Vérifier l'impact
print("Ancien score pluie (vigne) :", df.loc[mask_vigne, "score_pluie"].describe())
print("Nouveau score pluie (vigne) :", df.loc[mask_vigne, "score_pluie_corr"].describe())

df.to_csv("scores_cultures_corrige.csv", index=False, 
          columns=["maille_id", "culture", "annee", "precipitations_annuelles_mm",
                   "score_temperature", "score_pluie_corr", "score_ph", "score_global_corr"])

In [ ]:
for culture in df["culture"].unique():
    sub = df[df["culture"] == culture]
    print(f"\n{culture} :")
    print("  pluie_abs_min :", sub["pluie_absolue_min_mm_an"].unique())
    print("  pluie_abs_max :", sub["pluie_absolue_max_mm_an"].unique())
    print("  pluie_opt_min :", sub["pluie_optimale_min_mm_an"].unique())
    print("  pluie_opt_max :", sub["pluie_optimale_max_mm_an"].unique())
    print("  temp_abs_min  :", sub["temperature_absolue_min_c"].unique())
    print("  temp_abs_max  :", sub["temperature_absolue_max_c"].unique())

In [ ]:
# Créer une copie des scores corrigés pour la vigne, sinon on utilise les originaux
df["score_pluie_final"] = df["score_pluie"]
df["score_global_final"] = df["score_global"]
mask = df["culture"] == "Vigne"
df.loc[mask, "score_pluie_final"] = df.loc[mask, "score_pluie_corr"]
df.loc[mask, "score_global_final"] = df.loc[mask, "score_global_corr"]

# Statistiques par culture
for culture in ["Vigne", "Pois chiche", "Pomme de terre"]:
    sub = df[df["culture"] == culture]
    print(f"\n{culture} ({len(sub)} lignes)")
    print("--- score_temperature ---")
    print(sub["score_temperature"].describe())
    print("\n--- score_pluie_final ---")
    print(sub["score_pluie_final"].describe())
    print("\n--- score_ph ---")
    print(sub["score_ph"].describe())
    print("\n--- score_global_final ---")
    print(sub["score_global_final"].describe())

## Description du dataset `score_complet.csv`

Le fichier est produit dans le cadre du projet d'évaluation du potentiel agroclimatique.
Chaque ligne correspond à une **maille × culture × année**.

- **Résolution spatiale** : mailles CORDEX (1 pixel climat = 1 maille).  
  Le sol (pH) provient d'une source plus fine (SoilGrids) et est agrégé :  
  colonne `nombre_pixels_ph` (ex. 1653).
- **Période** : années présentes (au moins 2010–2024 dans l'extrait analysé).
- **Cultures** : Vigne, Pois chiche, Pomme de terre.
- **Variables environnementales** :
  - `temperature_moyenne_saison_c` : température moyenne durant la période de croissance (spécifique à chaque culture, ex. avril-octobre pour la vigne).
  - `precipitations_annuelles_mm` : précipitations annuelles totales.
  - `ph_moyen_60_100cm` : pH moyen du sol entre 60 et 100 cm de profondeur.
- **Seuils agronomiques** : issus de la base EcoCrop (FAO), avec lien de traçabilité dans `source_exigences`.  
  Colonnes de seuils absolus (min/max, score = 0) et optimaux (min/max, score = 100) pour chaque facteur.

## Méthodologie de calcul des scores

### Scores par facteur (`score_temperature`, `score_pluie`, `score_ph`)
Chaque sous-score est calculé par **interpolation linéaire continue** entre les bornes absolues et optimales :

- Dans la plage optimale → **100**
- Hors bornes absolues → **0**
- Entre absolue et optimale → interpolation (rampe linéaire de 0 à 100)

La formule est appliquée indépendamment pour la température, la pluie et le pH.

### Score global (`score_global`)
Le score global est une **moyenne pondérée** des trois sous-scores :
score_global = 0.6 × score_temperature + 0.2 × score_pluie + 0.2 × score_ph


⚠️ La pondération (60/20/20) n’est pas documentée dans le fichier et n’a pas encore été justifiée par une source agronomique.  
En l’absence de référence, une moyenne simple (33/33/33) serait plus facile à défendre.

## Problème identifié et correction (vigne)

**Constat** : 9 261 lignes Vigne avaient un `score_pluie = 0` malgré des précipitations annuelles réalistes (médiane 1362 mm).  
La cause : la **borne absolue maximale de pluie** pour la vigne était fixée à **1200 mm/an**, trop restrictive pour le contexte français.

**Correction appliquée** (en local, en attendant la mise à jour du fichier source par Ugo) :
- Nouvelle borne absolue max : **1500 mm/an** (cohérent avec la littérature, ex. EcoCrop / Gladstones)
- Nouvelles bornes optimales : **[600, 900] mm/an** (au lieu de [700, 850])

Le score pluie corrigé (`score_pluie_corr`) et le score global corrigé (`score_global_corr`) sont utilisés pour la vigne dans les analyses.  
Les autres cultures n’ont pas nécessité de correction (bornes déjà larges et cohérentes avec les valeurs observées).

## Distribution des scores après correction

| Culture        | Moy score global | Médiane | Principale contrainte                 |
|----------------|------------------|---------|---------------------------------------|
| Vigne          | 79.4             | 82.3    | Température (fraîcheur en altitude)   |
| Pois chiche    | 90.9             | 97.0    | Température (très rare, massifs)      |
| Pomme de terre | 90.6             | 93.5    | pH acide (sols granitiques/volcan.)  |

Les scores sont globalement élevés, ce qui reflète une large couverture des conditions optimales EcoCrop sur le territoire français. 
Les valeurs nulles résiduelles correspondent à des zones de haute montagne (température insuffisante) ou, pour la pomme de terre, à des sols très acides.

## Remarques importantes

- **pH** : statique dans le temps pour une maille donnée (le sol n’évolue pas d’une année sur l’autre).
- **Période de croissance** : variable selon la culture (ex. mars-août pour pois chiche, avril-octobre pour vigne).
- **Traceabilité** : chaque ligne référence l’URL de la fiche EcoCrop utilisée (`source_exigences`).
- **Pixels** : `nombre_pixels_climat = 1` (CORDEX natif), `nombre_pixels_ph` >> 1 (agrégation depuis une couche sol plus fine).

Le jeu de données corrigé est sauvegardé dans `score_vigne_corrige.csv`.

In [ ]:
%pip install plotly

In [ ]:
%pip install kaleido

In [ ]:
%pip install nbformat

In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install nbformat --upgrade

In [ ]:
import pandas as pd
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")  # adapte le nom si nécessaire

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Charger les données
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")

# 2. Fonction de score linéaire
def compute_linear_score(x, abs_min, opt_min, opt_max, abs_max):
    if x <= abs_min or x >= abs_max:
        return 0.0
    if opt_min <= x <= opt_max:
        return 100.0
    if abs_min < x < opt_min:
        return (x - abs_min) / (opt_min - abs_min) * 100
    if opt_max < x < abs_max:
        return (abs_max - x) / (abs_max - opt_max) * 100
    return 0.0

# 3. Correction des bornes pour la vigne
new_abs_min = 400
new_abs_max = 1500
new_opt_min = 600
new_opt_max = 900

mask_vigne = df["culture"] == "Vigne"
df["score_pluie_corr"] = df["score_pluie"]  # copie initiale
df.loc[mask_vigne, "score_pluie_corr"] = df.loc[mask_vigne, "precipitations_annuelles_mm"].apply(
    lambda x: compute_linear_score(x, new_abs_min, new_opt_min, new_opt_max, new_abs_max)
)

df["score_global_corr"] = df["score_global"]
df.loc[mask_vigne, "score_global_corr"] = (
    0.6 * df.loc[mask_vigne, "score_temperature"] +
    0.2 * df.loc[mask_vigne, "score_pluie_corr"] +
    0.2 * df.loc[mask_vigne, "score_ph"]
)

# 4. Agréger par maille (moyenne sur les années) pour la vigne
vigne = df[df["culture"] == "Vigne"].copy()
vigne_agg = vigne.groupby(["maille_id", "longitude_centre", "latitude_centre"]).agg(
    score_moyen=("score_global_corr", "mean"),
    temp_moy=("temperature_moyenne_saison_c", "mean"),
    pluie_moy=("precipitations_annuelles_mm", "mean"),
    ph_moy=("ph_moyen_60_100cm", "mean")
).reset_index()

# 5. Créer la carte
fig = px.scatter_map(
    vigne_agg,
    lat="latitude_centre",
    lon="longitude_centre",
    color="score_moyen",
    color_continuous_scale="RdYlGn",
    range_color=(0, 100),
    size_max=10,
    zoom=4,
    map_style="carto-positron",
    hover_data={
        "maille_id": True,
        "score_moyen": ":.1f",
        "temp_moy": ":.1f",
        "pluie_moy": ":.0f",
        "ph_moy": ":.2f"
    },
    title="Score global moyen (vigne) – correction 1500 mm"
)

# 6. Affichage avec fallback navigateur si nbformat absent
try:
    import nbformat
    fig.show()
except ImportError:
    fig.show(renderer="browser")

In [ ]:
import plotly.express as px

# Fonction pour générer la carte d'une culture donnée
def carte_score_global(culture_nom, colonne_score="score_global"):
    sub = df[df["culture"] == culture_nom].copy()
    agg = sub.groupby(["maille_id", "longitude_centre", "latitude_centre"]).agg(
        score_moyen=(colonne_score, "mean"),
        temp_moy=("temperature_moyenne_saison_c", "mean"),
        pluie_moy=("precipitations_annuelles_mm", "mean"),
        ph_moy=("ph_moyen_60_100cm", "mean")
    ).reset_index()
    
    fig = px.scatter_map(
        agg,
        lat="latitude_centre",
        lon="longitude_centre",
        color="score_moyen",
        color_continuous_scale="RdYlGn",
        range_color=(0, 100),
        size_max=10,
        zoom=4,
        map_style="carto-positron",
        hover_data={
            "maille_id": True,
            "score_moyen": ":.1f",
            "temp_moy": ":.1f",
            "pluie_moy": ":.0f",
            "ph_moy": ":.2f"
        },
        title=f"Score global moyen – {culture_nom}"
    )
    return fig

# Pois chiche
fig_pois = carte_score_global("Pois chiche")
fig_pois.show()

# Pomme de terre
fig_pdt = carte_score_global("Pomme de terre")
fig_pdt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Charger les données
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")

# 2. Fonction de score linéaire
def compute_linear_score(x, abs_min, opt_min, opt_max, abs_max):
    if x <= abs_min or x >= abs_max:
        return 0.0
    if opt_min <= x <= opt_max:
        return 100.0
    if abs_min < x < opt_min:
        return (x - abs_min) / (opt_min - abs_min) * 100
    if opt_max < x < abs_max:
        return (abs_max - x) / (abs_max - opt_max) * 100
    return 0.0

# 3. Unification des scores (Correction Vigne + Score standard pour les autres)
df["score_final"] = df["score_global"] # Par défaut, on garde le score initial

mask_vigne = df["culture"] == "Vigne"
score_pluie_vigne = df.loc[mask_vigne, "precipitations_annuelles_mm"].apply(
    lambda x: compute_linear_score(x, 400, 600, 900, 1500)
)

# Application de la pondération uniquement sur la Vigne
df.loc[mask_vigne, "score_final"] = (
    0.6 * df.loc[mask_vigne, "score_temperature"] +
    0.2 * score_pluie_vigne +
    0.2 * df.loc[mask_vigne, "score_ph"]
)

# 4. Filtrer sur les cultures cibles et Agréger
cultures_cibles = ["Vigne", "Pois chiche", "Pomme de terre"]
df_cibles = df[df["culture"].isin(cultures_cibles)]

agg = df_cibles.groupby(["culture", "maille_id", "longitude_centre", "latitude_centre"]).agg(
    score_moyen=("score_final", "mean"),
    temp_moy=("temperature_moyenne_saison_c", "mean"),
    pluie_moy=("precipitations_annuelles_mm", "mean"),
    ph_moy=("ph_moyen_60_100cm", "mean")
).reset_index()

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 5. Créer l'espace avec 3 colonnes de type "map"
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Vigne", "Pois chiche", "Pomme de terre"),
    specs=[[{"type": "map"}, {"type": "map"}, {"type": "map"}]]
)

cultures = ["Vigne", "Pois chiche", "Pomme de terre"]

# Boucle pour ajouter chaque culture dans sa colonne respective
for col_idx, culture in enumerate(cultures, start=1):
    sub_df = agg[agg["culture"] == culture]
    
    fig.add_trace(
        go.Scattermap(
            lat=sub_df["latitude_centre"],
            lon=sub_df["longitude_centre"],
            mode="markers",
            marker=dict(
                color=sub_df["score_moyen"],
                colorscale="RdYlGn",
                cmin=0, cmax=100,
                size=8,
                showscale=(col_idx == 3), # Affiche la légende une seule fois sur la 3ème carte
                colorbar=dict(title="Score") if col_idx == 3 else None
            ),
            text=sub_df.apply(lambda row: f"Maille: {row['maille_id']}<br>Score: {row['score_moyen']:.1f}<br>Temp: {row['temp_moy']:.1f}°C<br>Pluie: {row['pluie_moy']:.0f}mm", axis=1),
            hovertemplate="%{text}<extra></extra>",
            name=culture
        ),
        row=1, col=col_idx
    )

# 6. Configurer le fond de carte et le zoom pour les 3 cartes
map_layout = dict(style="carto-positron", center=dict(lat=46.5, lon=2.5), zoom=4)

fig.update_layout(
    title_text="Comparaison des scores par culture (2010-2025)",
    map=map_layout,
    map2=map_layout,
    map3=map_layout,
    margin={"r":0, "t":50, "l":0, "b":0}
)

# Affichage
try:
    import nbformat
    fig.show()
except ImportError:
    fig.show(renderer="browser")

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Chargement (à exécuter une seule fois si ce n'est pas déjà fait)
df = pd.read_csv("scores_cultures_mailles_12km_2010_2025.csv")

In [ ]:
def compute_linear_score(x, abs_min, opt_min, opt_max, abs_max):
    if x <= abs_min or x >= abs_max:
        return 0.0
    if opt_min <= x <= opt_max:
        return 100.0
    if abs_min < x < opt_min:
        return (x - abs_min) / (opt_min - abs_min) * 100
    if opt_max < x < abs_max:
        return (abs_max - x) / (abs_max - opt_max) * 100
    return 0.0

new_abs_min, new_abs_max = 400, 1500
new_opt_min, new_opt_max = 600, 900

mask_vigne = df["culture"] == "Vigne"
if "score_pluie_corr" not in df.columns:
    df["score_pluie_corr"] = df["score_pluie"]
    df.loc[mask_vigne, "score_pluie_corr"] = df.loc[mask_vigne, "precipitations_annuelles_mm"].apply(
        lambda x: compute_linear_score(x, new_abs_min, new_opt_min, new_opt_max, new_abs_max)
    )
    df["score_global_corr"] = df["score_global"]
    df.loc[mask_vigne, "score_global_corr"] = (
        0.6 * df.loc[mask_vigne, "score_temperature"] +
        0.2 * df.loc[mask_vigne, "score_pluie_corr"] +
        0.2 * df.loc[mask_vigne, "score_ph"]
    )

In [ ]:
# Préparer une colonne score_global unifiée : corrigé pour la vigne, original pour les autres
df["score_global_final"] = df["score_global"]
df.loc[mask_vigne, "score_global_final"] = df.loc[mask_vigne, "score_global_corr"]

serie = df.groupby(["culture", "annee"])["score_global_final"].mean().reset_index()

fig = px.line(
    serie,
    x="annee",
    y="score_global_final",
    color="culture",
    markers=True,
    title="Évolution du score global moyen (toutes mailles) – Vigne avec correction 1500 mm",
    labels={"score_global_final": "Score global moyen", "annee": "Année"}
)
fig.show()

In [ ]:
df["score_pluie_final"] = df["score_pluie"]
df.loc[mask_vigne, "score_pluie_final"] = df.loc[mask_vigne, "score_pluie_corr"]

df_melt = df.melt(
    id_vars=["culture"],
    value_vars=["score_temperature", "score_pluie_final", "score_ph"],
    var_name="Facteur",
    value_name="Score"
)

df_melt["Facteur"] = df_melt["Facteur"].replace({
    "score_temperature": "Température",
    "score_pluie_final": "Pluie (corrigé)",
    "score_ph": "pH"
})

fig = px.box(
    df_melt,
    x="culture",
    y="Score",
    color="Facteur",
    title="Distribution des sous-scores par culture (vigne avec correction pluie)",
    points=False
)
fig.show()

In [ ]:
sample = df[df["culture"] == "Vigne"].sample(3000, random_state=1)

fig = px.scatter(
    sample,
    x="score_temperature",
    y="score_pluie_corr",
    color="score_global_corr",
    color_continuous_scale="RdYlGn",
    range_color=[0,100],
    title="Vigne : score température vs score pluie (corrigé) – Couleur = score global",
    hover_data=["maille_id", "annee"]
)
fig.add_hline(y=100, line_dash="dash", line_color="grey", opacity=0.5)
fig.add_vline(x=100, line_dash="dash", line_color="grey", opacity=0.5)
fig.show()

In [ ]:
vigne = df[df["culture"] == "Vigne"].copy()
plot_df = vigne[["score_pluie", "score_pluie_corr"]].melt(
    var_name="Version", value_name="Score"
)
plot_df["Version"] = plot_df["Version"].replace({
    "score_pluie": "Avant (max 1200 mm)",
    "score_pluie_corr": "Après (max 1500 mm)"
})

fig = px.histogram(
    plot_df, x="Score", color="Version", nbins=50,
    opacity=0.7, barmode="overlay",
    title="Impact de la correction de la borne pluie (vigne)",
    color_discrete_map={"Avant (max 1200 mm)": "blue", "Après (max 1500 mm)": "red"}
)
fig.show()

In [ ]:
cultures = df["culture"].unique()
moyennes = {}
for c in cultures:
    sub = df[df["culture"] == c]
    temp = sub["score_temperature"].mean()
    pluie = sub["score_pluie_final"].mean() if c == "Vigne" else sub["score_pluie"].mean()
    ph = sub["score_ph"].mean()
    moyennes[c] = [temp, pluie, ph]

fig = go.Figure()
for c in cultures:
    fig.add_trace(go.Scatterpolar(
        r=moyennes[c],
        theta=["Température", "Pluie", "pH"],
        fill="toself",
        name=c
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,100])),
    title="Profil moyen des sous-scores par culture"
)
fig.show()

In [ ]:
maille_id = "M0005"  # change selon tes besoins
sub = df[(df["maille_id"] == maille_id) & (df["culture"] == "Vigne")]

fig = px.imshow(
    [sub["score_global_corr"].values],
    x=sub["annee"],
    y=["Score global"],
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=100,
    aspect="auto",
    title=f"Évolution du score global vigne – maille {maille_id}"
)
fig.update_xaxes(side="top", dtick=1)
fig.show()